# Setup

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
import sys

In [ ]:
base_path = 'data/imputation_quality'

In [ ]:
def comp_plot(df, x, y, title=None):
    g = sns.scatterplot(data=df, x=x, y=y)
    stat_out = stats.spearmanr(df[x], df[y])
    plt.text(0.05, 0.95, 'Spearman\'s', transform=plt.gca().transAxes)
    # figtext = 'rho={:.4f}'.format(stat_out.statistic) + '; p={:.2e}; '.format(stat_out.pvalue) + f'n={len(df)}'
    figtext = 'rho={:.4f}'.format(stat_out.statistic) + f'; n={len(df)}'
    # plt.title('imputed vs genotyped allele frequencies, manually curated')
    if title is not None:
        plt.title(title)
    plt.text(0.1, 0.9, figtext, transform=plt.gca().transAxes)
    plt.tick_params(left=False, bottom=False)
    plt.tight_layout()

def hexbin_plot(df, x, y, title=None):
    hb = plt.hexbin(df[x], df[y], gridsize=15, cmap='Blues')
    cb = plt.colorbar(hb)
    plt.xlabel(x)
    plt.ylabel(y)
    stat_out = stats.spearmanr(df[x], df[y])
    plt.text(0.05, 0.95, 'Spearman\'s', transform=plt.gca().transAxes)
    # figtext = 'rho={:.4f}'.format(stat_out.statistic) + '; p={:.2e}; '.format(stat_out.pvalue) + f'n={len(df)}'
    figtext = 'rho={:.4f}'.format(stat_out.statistic) + f'; n={len(df)}'
    # plt.title('imputed vs genotyped allele frequencies, manually curated')
    if title is not None:
        plt.title(title)
    plt.text(0.1, 0.9, figtext, transform=plt.gca().transAxes)
    plt.tick_params(left=False, bottom=False)
    plt.tight_layout()


def plot_allele_length_distribution(df, title=None):
    p = sns.histplot(data=df, x='r^2', bins=20, stat='percent')
    if title is None:
        title = 'Distribution of Pearson Correlation of Sum of Allele Lengths per Locus; n={len(df)}'
    plt.title(title)
    plt.tick_params(left=False, bottom=False)
    plt.xlabel(r'Pearson Correlation Coefficient $r^2$')
    return p
# t.savefig(f'{base_path}/combined_corrs_r2_hist.png', dpi=300)

In [ ]:
def get_results(file_path, is_meta=False, cut=True, cut_threshold=10):
    df = pd.read_csv(file_path, sep="\t")
    initial = df.shape[0]
    df = df.dropna(axis=0)
    after = df.shape[0]
    print(f'drop {initial-after} rows because of NA values')
    df = df.astype({'P':float})
    df.loc[df.P<sys.float_info.min, 'P']=sys.float_info.min

    min_ct = df[df.P == sys.float_info.min].shape[0]
    if min_ct > 0:
        print(f'set {min_ct} P values to maximum possible min of {sys.float_info.min}')
    df['-log10'] = -np.log10(df.P)

    if not is_meta:
        df = df[df['TEST']=='ADD']

    if cut:
        df=df[df['-log10'] < cut_threshold]
    
    return df

In [ ]:
strs_to_rm1 = pd.read_excel('data/str_filtering/non_strs.ods', sheet_name='suggestive')
strs_to_rm2 = pd.read_excel('data/str_filtering/non_strs.ods', sheet_name='gw')
strs_to_rm = pd.concat([strs_to_rm1, strs_to_rm2], ignore_index=True)

## Match variants

In [ ]:
istrs = get_results("data/imputed_strs/subset/white_british_combined_chr_strsub.glm.linear", cut=False)
istrs = istrs.drop(istrs[istrs.ID.isin(strs_to_rm.ID)].index)
gstrs = get_results("data/gSTR/gSTR_white_british_assoc.ad_risk_by_proxy.glm.linear", cut=False)
gstr_motifs = pd.read_csv('data/gSTR/gSTR_motifs.tsv', sep='\t')
gstrs = gstrs.merge(gstr_motifs, on='ID', how='left')

In [ ]:
# calculate allele length
gstrs['ref_len'] = gstrs['REF'].map(lambda x: len(x))
gstrs['mot_num'] = gstrs['ALT'].str.extract(r'<([\d.]+)>').astype(float)
gstrs['mot_len'] = gstrs['Motif'].str.len()
gstrs['alt_len'] = gstrs['mot_num'] * gstrs['mot_len']
gstrs['alt_len'] = gstrs['alt_len'].astype(int)

In [ ]:
# merge by position and allele lengths
merge_alleles = istrs.merge(
    gstrs, how='inner', 
    on=['#CHROM', 'POS', 'ref_len', 'alt_len'], 
    suffixes=('_imputed', '_genotyped')
)
print(f'Merged dataset by position and allele lengths has {merge_alleles.shape[0]} variants')
# save matching id pairs
merge_alleles[['ID_genotyped', 'ID_imputed']].to_csv(f'{base_path}/matched_alleles.tsv', sep='\t', index=False)

# Comparison of genotyped vs imputed STR genotypes

## Automatic matching of ~600K variants

### Betas

In [ ]:
comp_plot(merge_alleles, x='BETA_imputed', y='BETA_genotyped', title=None)

### Allele Frequencies

In [ ]:
# Combine allele frequencies to the corresponding ids
istr_freqs = pd.read_csv(f'{base_path}/check_freqs/iSTR_freqs_sub.afreq', sep='\t')
gstr_freqs = pd.read_csv(f'{base_path}/check_freqs/gSTR_freqs.afreq', sep='\t')
ids = merge_alleles[['ID_imputed', 'ID_genotyped']].drop_duplicates()
freqs_merged = ids.merge(istr_freqs, left_on='ID_imputed', right_on='ID', how='left')
freqs_merged = freqs_merged.merge(gstr_freqs, left_on='ID_genotyped', right_on='ID', how='left', suffixes=('_imputed', '_genotyped'))
freqs_merged = freqs_merged.rename(columns={col: col.replace('_x', '_imputed') for col in freqs_merged.columns if col.endswith('_x')})
freqs_merged = freqs_merged.rename(columns={col: col.replace('_y', '_genotyped') for col in freqs_merged.columns if col.endswith('_y')})
freqs_merged = freqs_merged.drop(columns=['#ID_imputed', '#ID_genotyped'])

In [ ]:
# split frequencies into reference and alternative
freqs_merged['Reference Frequency (genotyped)'] = freqs_merged['FREQS_genotyped'].map(lambda x: float(x.split(',')[0]) if isinstance(x, str) else x)
freqs_merged['Reference Frequency (imputed)'] = freqs_merged['FREQS_imputed'].map(lambda x: float(x.split(',')[0]) if isinstance(x, str) else x)
freqs_merged['Alternative Frequency (genotyped)'] = freqs_merged['FREQS_genotyped'].map(lambda x: float(x.split(',')[1]) if isinstance(x, str) else x)
freqs_merged['Alternative Frequency (imputed)'] = freqs_merged['FREQS_imputed'].map(lambda x: float(x.split(',')[1]) if isinstance(x, str) else x)

In [ ]:
# Do a hexbin plot since the scatter plot is too crowded
hexbin_plot(freqs_merged, 'Reference Frequency (genotyped)', 'Reference Frequency (imputed)')

### Allele lengths
In order to run this, read the *Correlation of Allele Lengths* part of the README and run the respective scripts. 

In [ ]:
header = ['imputed_variant_id', 'genotyped_variant_id', 'corr', 'n']
df_big = pd.read_csv(f'{base_path}/600K_combined_corrs.tsv', sep='\t', names=header)
df_big['r^2'] = df_big['corr']**2

In [ ]:
title = 'Distribution of Pearson Correlation of Sum of Allele Lengths per Locus\n' + \
    f'Matches by position and allele length, n={len(df_big)}'
plot_allele_length_distribution(df_big, title=title)

In [ ]:
df_big[['corr', 'r^2']].describe()

In [ ]:
print(df_big["r^2"].describe().apply(lambda x: f"{x:.7f}"))

In [ ]:
(df_big[['corr', 'r^2']]>0.8).mean()

In [ ]:
(df_big[['corr', 'r^2']]>0.9).mean()

## Manual matching of curated variants
This follows the same logic, only that we need to load the manually matched variant-pairs